In [4]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from Bio import Entrez
import pandas as pd
import time
from src.fetcher import configure_entrez

configure_entrez
print("ready")

ready


In [5]:
configure_entrez()

Entrez configured with email: akharya@ucsd.edu


In [6]:
from Bio import Entrez

pubmed_query = """ 
("gut microbiome" OR "gut microbiota" OR "intestinal microbiome" OR 
"gut metagenome" OR "fecal microbiome" OR "faecal microbiome") 
AND 
("shotgun" OR "metagenomics" OR "whole genome sequencing" OR 
"whole metagenome sequencing" OR "metagenomic sequencing")
AND
("animal" OR "mammal" OR "bird" OR "fish" OR "reptile" OR "amphibian" OR 
"wildlife" OR "bovine" OR "porcine" OR "avian" OR "murine" OR "non-human")
"""

handle = Entrez.esearch(
    db = 'pubmed', 
    term = pubmed_query, 
    retmax = 500, 
    usehistory = 'y'
)

record = Entrez.read(handle)
handle.close()

print(f"PubMed papers found: {record['Count']}")
print(f"Retrieved: {len(record['IdList'])} IDs")

PubMed papers found: 1853
Retrieved: 500 IDs


In [7]:
# fetch abstracts and extract ENA/SRA accession numbers
from Bio import Entrez
import re

accession_pattern = re.compile(
    r'\b(PRJ[EDN][A-Z]\d+|[EDS]RP\d+|[EDS]RR\d+)\b'
)

pubmed_ids = record["IdList"]
found_accessions = set()

print(f"Searching {len(pubmed_ids)} papers for ENA/SRA accessions...")

# fetch in batches of 50
batch_size = 50
for i in range(0, len(pubmed_ids), batch_size):
    batch = pubmed_ids[i:i+batch_size]
    handle = Entrez.efetch(
        db="pubmed",
        id=",".join(batch),
        rettype="abstract",
        retmode="text"
    )
    text = handle.read()
    handle.close()
    
    matches = accession_pattern.findall(text)
    found_accessions.update(matches)
    
    print(f"Batch {i//batch_size + 1}: found {len(matches)} accessions so far ({len(found_accessions)} unique)")
    time.sleep(0.5)

print(f"\nTotal unique accessions found: {len(found_accessions)}")

Searching 500 papers for ENA/SRA accessions...
Batch 1: found 0 accessions so far (0 unique)
Batch 2: found 0 accessions so far (0 unique)
Batch 3: found 1 accessions so far (1 unique)
Batch 4: found 0 accessions so far (1 unique)
Batch 5: found 0 accessions so far (1 unique)
Batch 6: found 0 accessions so far (1 unique)
Batch 7: found 0 accessions so far (1 unique)
Batch 8: found 0 accessions so far (1 unique)
Batch 9: found 0 accessions so far (1 unique)
Batch 10: found 0 accessions so far (1 unique)

Total unique accessions found: 1


In [8]:
# look at one abstract to see what we're working with
handle = Entrez.efetch(
    db="pubmed",
    id=pubmed_ids[0],
    rettype="abstract",
    retmode="text"
)
text = handle.read()
handle.close()
print(text)

1. Nutr J. 2026 May 25. doi: 10.1186/s12937-026-01335-5. Online ahead of print.

Gut microbiota and western dietary patterns associated with behavioral problems 
in children and adolescents: a cross-sectional study.

Larroya A(#)(1), Romera-Giner S(#)(1), Tolosa-Enguís V(1), Rodríguez-Ruano 
SM(1), Andrés-García S(2), Soro-Conde I(2), Codoñer P(3)(4), Sanz Y(5).

Author information:
(1)Microbiome Innovation in Nutrition & Health, Institute of Agrochemistry and 
Food Technology, Spanish National Research Council (IATA-CSIC), Avda. Agustin 
Escardino, 7, Valencia, Paterna, 46980, Spain.
(2)Neurological Rehabilitation Center, Limbic, Valencia, 46015, Spain.
(3)Deparment of Pediatrics, Dr. Peset University Hospital, Avd. De Gaspar 
Aguilar, 80, Valencia, 46017, Spain.
(4)Deparment of Pediatrics, Obstetrics and Gynecology, University of Valencia, 
Valencia, Spain.
(5)Microbiome Innovation in Nutrition & Health, Institute of Agrochemistry and 
Food Technology, Spanish National Research Counc

In [9]:
# try ENA's xref endpoint 
import requests

pubmed_id = "42075321"

response = requests.get(
    f"https://www.ebi.ac.uk/ena/xref/rest/json/search?source=PubMed&source_primary_id={pubmed_id}",
    timeout=30
)
print("Status:", response.status_code)
print("Response:", response.text[:500])

Status: 200
Response: [{"Source":"PubMed","Source Primary Accession":"11407914","Source Secondary Accession":"","Source URL":"http://europepmc.org/abstract/MED/11407914","Source Secondary URL":"","Target":"study","Target Primary Accession":"PRJNA271","Target Secondary Accession":"","Target URL":"https://www.ebi.ac.uk/ena/browser/view/PRJNA271","Has Inferred":"N","Inferred From":""},{"Source":"PubMed","Source Primary Accession":"11408212","Source Secondary Accession":"","Source URL":"http://europepmc.org/abstract/MED/


In [10]:
# map PubMed IDs to ENA study accessions via xref
pubmed_to_ena = {}

print(f"Mapping {len(pubmed_ids)} PubMed IDs to ENA accessions...")

for i, pmid in enumerate(pubmed_ids):
    if i % 50 == 0:
        print(f"Progress: {i}/{len(pubmed_ids)}")
    
    try:
        response = requests.get(
            f"https://www.ebi.ac.uk/ena/xref/rest/json/search?source=PubMed&source_primary_id={pmid}",
            timeout=30
        )
        if response.status_code == 200 and response.json():
            for record in response.json():
                if record.get("Target") == "study":
                    ena_acc = record.get("Target Primary Accession")
                    if ena_acc:
                        pubmed_to_ena[pmid] = ena_acc
                        break
    except Exception as e:
        print(f"Error for {pmid}: {e}")
    
    time.sleep(0.2)

print(f"\nFound ENA accessions for {len(pubmed_to_ena)} out of {len(pubmed_ids)} papers")
print(f"Unique ENA studies: {len(set(pubmed_to_ena.values()))}")

Mapping 500 PubMed IDs to ENA accessions...
Progress: 0/500


KeyboardInterrupt: 

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from Bio import Entrez
import time
import pandas as pd
from src.fetcher import configure_entrez

configure_entrez
print("ready")

ready


In [ ]:
from src.ena_fetcher import fetch_pubmed_abstract_by_title

In [ ]:
import importlib 
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import fetch_pubmed_abstract_by_title
from src.fetcher import configure_entrez

configure_entrez()

Entrez configured with email: akharya@ucsd.edu


In [ ]:
df = pd.read_csv("../results/host_enriched_462studies.csv")

has_title = df[df['title'].notna()].head(5)

In [ ]:
for i, row in has_title.iterrows():
    abstract = fetch_pubmed_abstract_by_title(row["title"])
    print(f"\n{row['study_accession']}")
    print(f"Title: {row['title'][:100]}")
    print(f"Abstract: {'FOUND' if abstract else 'NOT FOUND'}")


PRJNA801645
Title: Study of the effects of a stabilizer on microbiome analysis
Abstract: NOT FOUND

PRJEB47613
Title: HoloFood Chicken Caecum+Ileum Metagenome – non-targeted metabolomics batch
Abstract: NOT FOUND

PRJDB10675
Title: International collaborative studies on the prevention, diagnosis, and drug discovery for diarrheal i
Abstract: NOT FOUND

PRJNA835973
Title: Lactobacillus rhamnosus Probio-M9 alleviates colitis-associated tumorigenesis via modified gut micro
Abstract: NOT FOUND

PRJEB86258
Title: 3D'omics: Salmonella challenge chicken trial
Abstract: NOT FOUND


In [ ]:
from Bio import Entrez

# test one title manually
title = has_title.iloc[0]["title"]
print(f"Searching for: {title}")

handle = Entrez.esearch(
    db="pubmed",
    term=f'"{title}"[Title]',
    retmax=5
)
record = Entrez.read(handle)
handle.close()

print(f"Count: {record['Count']}")
print(f"IDs: {record['IdList']}")

Searching for: Study of the effects of a stabilizer on microbiome analysis
Count: 0
IDs: []


In [ ]:
# try fuzzy search instead of exact title match
title = has_title.iloc[0]["title"]
print(f"Original title: {title}")

# try without quotes - broader search
handle = Entrez.esearch(
    db="pubmed",
    term=f'{title}[Title]',
    retmax=3
)
record = Entrez.read(handle)
handle.close()

print(f"Fuzzy search count: {record['Count']}")
print(f"IDs: {record['IdList']}")

# also try searching just key words
words = " AND ".join(title.split()[:4])
handle2 = Entrez.esearch(
    db="pubmed",
    term=words,
    retmax=3
)
record2 = Entrez.read(handle2)
handle2.close()

print(f"\nKeyword search: {words}")
print(f"Count: {record2['Count']}")
print(f"IDs: {record2['IdList']}")

Original title: Study of the effects of a stabilizer on microbiome analysis
Fuzzy search count: 5
IDs: ['41280693', '39256748', '38102642']

Keyword search: Study AND of AND the AND effects
Count: 0
IDs: []


In [ ]:
from Bio import Entrez
import time
from src.fetcher import configure_entrez

configure_entrez()

pubmed_query = """
("gut microbiome" OR "gut microbiota" OR "intestinal microbiome" OR 
"gut metagenome" OR "fecal microbiome" OR "faecal microbiome") 
AND 
("shotgun" OR "metagenomics" OR "whole genome sequencing" OR 
"whole metagenome sequencing" OR "metagenomic sequencing")
AND
("animal" OR "mammal" OR "bird" OR "fish" OR "reptile" OR "amphibian" OR 
"wildlife" OR "bovine" OR "porcine" OR "avian" OR "murine" OR "non-human")
"""

handle = Entrez.esearch(
    db="pubmed",
    term=pubmed_query,
    retmax=500,
    usehistory="y"
)
record = Entrez.read(handle)
handle.close()

print(f"PubMed papers found: {record['Count']}")
print(f"Retrieved: {len(record['IdList'])} IDs")



Entrez configured with email: akharya@ucsd.edu
PubMed papers found: 1843
Retrieved: 500 IDs


In [ ]:
with open("../results/pubmed_ids.txt", "w") as f:
    for pmid in record["IdList"]:
        f.write(pmid + "\n")

print(f"Saved to results/pubmed_ids.txt")

Saved to results/pubmed_ids.txt


In [ ]:
# post using Alan's MMC tool
import pandas as pd 
import re 

existing = pd.read_csv("../results/host_enriched_462studies.csv")
existing_accessions = set(existing["study_accession"].tolist())

# load MMC results
mmc = pd.read_csv("../results/pmid-ena-results-2026-05-18T00-45-37-328Z.csv")

# extract all project accessions
new_accessions = []
for acc_str in mmc[mmc["accessions"].notna()]["accessions"]:
    for acc in acc_str.split(";"):
        if re.match(r'PRJ[ENDB][A-Z]\d+', acc) and acc not in existing_accessions:
            new_accessions.append(acc)

new_accessions = list(set(new_accessions))
print(f"New studies not in existing catalog: {len(new_accessions)}")

New studies not in existing catalog: 239


In [ ]:
# get remaining IDs starting from offset 500
from Bio import Entrez
import time
from src.fetcher import configure_entrez

configure_entrez()


pubmed_query = """
("gut microbiome" OR "gut microbiota" OR "intestinal microbiome" OR 
"gut metagenome" OR "fecal microbiome" OR "faecal microbiome") 
AND 
("shotgun" OR "metagenomics" OR "whole genome sequencing" OR 
"whole metagenome sequencing" OR "metagenomic sequencing")
AND
("animal" OR "mammal" OR "bird" OR "fish" OR "reptile" OR "amphibian" OR 
"wildlife" OR "bovine" OR "porcine" OR "avian" OR "murine" OR "non-human")
"""


handle = Entrez.esearch(
    db="pubmed",
    term=pubmed_query,
    retstart=500,  # skip first 500
    retmax=1500,   # get the rest
    usehistory="y"
)
record2 = Entrez.read(handle)
handle.close()

print(f"Retrieved: {len(record2['IdList'])} additional IDs")

# save to separate file
with open("../results/pubmed_ids_batch2.txt", "w") as f:
    for pmid in record2["IdList"]:
        f.write(pmid + "\n")

print(f"Saved to pubmed_ids_batch2.txt")

Entrez configured with email: akharya@ucsd.edu
Retrieved: 1346 additional IDs
Saved to pubmed_ids_batch2.txt


In [ ]:
import importlib 
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import fetch_pubmed_abstract_by_title,\
    resolve_host_species,fetch_runs_for_study
from src.fetcher import configure_entrez

configure_entrez()

Entrez configured with email: akharya@ucsd.edu


In [ ]:
new_results = []

for i, study in enumerate(new_accessions):
    if i % 20 == 0:
        print(f"Progress: {i}/{len(new_accessions)}")
    
    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue
    
    new_results.append({
        "study_accession": study,
        "host_species": resolve_host_species(runs_df),
        "host_tax_id": runs_df["host_tax_id"].dropna().mode()[0] if len(runs_df["host_tax_id"].dropna()) > 0 else None,
        "body_site": runs_df["host_body_site"].dropna().mode()[0] if len(runs_df["host_body_site"].dropna()) > 0 else "",
        "country": runs_df["country"].dropna().mode()[0] if len(runs_df["country"].dropna()) > 0 else "",
        "n_samples": runs_df["sample_accession"].nunique(),
        "library_strategy": runs_df["library_strategy"].dropna().mode()[0] if len(runs_df["library_strategy"].dropna()) > 0 else None,
    })
    time.sleep(0.3)

print(f"\nFetched metadata for {len(new_results)} new studies")

Progress: 0/239
No runs found for PRJNA1126667
No runs found for PRJNA1381256
No runs found for PRJNA1416997
No runs found for PRJNA1030165
Progress: 20/239
No runs found for PRJNA10203991
No runs found for PRJNA1264687
No runs found for PRJNA1418170
No runs found for PRJEB801559
No runs found for PRJNA1049387
Progress: 40/239
No runs found for PRJNA1144017
No runs found for PRJNA1431711
No runs found for PRJNA1344668
Progress: 60/239
No runs found for PRJNA1322061
No runs found for PRJNA1227685
No runs found for PRJNA1024674
No runs found for PRJNA128711
No runs found for PRJNA1289876
Progress: 80/239
No runs found for PRJNA1333881
No runs found for PRJNA687329
Progress: 100/239
Progress: 120/239
No runs found for PRJNA1428989
No runs found for PRJNA1205488
No runs found for PRJNA977461
No runs found for PRJNA1196159
Progress: 140/239
No runs found for PRJEB94130
No runs found for PRJEB55374
No runs found for PRJNA1217447
No runs found for PRJNA1224300
Progress: 160/239
No runs found 

In [ ]:
import pandas as pd

# existing catalog
existing_df = pd.read_csv("../results/host_enriched_462studies.csv")

# new studies
new_df = pd.DataFrame(new_results)

# keep only columns that exist in both
common_cols = ["study_accession", "host_species", "host_tax_id", 
               "body_site", "country", "n_samples", "library_strategy"]

existing_aligned = existing_df[[c for c in common_cols if c in existing_df.columns]]
new_aligned = new_df[[c for c in common_cols if c in new_df.columns]]

# merge and deduplicate
combined_df = pd.concat([existing_aligned, new_aligned], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=["study_accession"])

print(f"Existing studies: {len(existing_df)}")
print(f"New studies: {len(new_df)}")
print(f"Combined unique studies: {len(combined_df)}")

NameError: name 'new_results' is not defined

In [ ]:
from src.ena_fetcher import fetch_abstract_from_pmid

In [ ]:
abstract = fetch_abstract_from_pmid("42121260")  # from Alan's CSV
print(abstract[:600] if abstract else "NOT FOUND")

1. Microbiome. 2026 May 12;14(1):144. doi: 10.1186/s40168-026-02369-x.

Integrative analysis of the mouse cecal microbiome across diet, age, and weight 
in the diverse BXD population.

Zhou Z(1), Lamanna A(1), Halder R(1), Pansart E(1), Narayanasamy S(1), Boussoufa 
B(1), Kerkour T(2), Wilmes P(1), Williams E(3).

Author information:
(1)Luxembourg Centre for Systems Biomedicine, University of Luxembourg, 
Esch-Sur-Alzette, 4362, Luxembourg.
(2)Department of Dermatology, Erasmus MC Cancer Institute, Rotterdam, 3015 GD, 
The Netherlands.
(3)Luxembourg Centre for Systems Biomedicine, University o


In [ ]:
import pandas as pd
import re

mmc_df = pd.read_csv("../results/pmid-ena-results-2026-05-18T00-45-37-328Z.csv")
print(f"Loaded {len(mmc_df)} papers")
print(f"Papers with accessions: {(mmc_df['accession_count'] > 0).sum()}")

FileNotFoundError: [Errno 2] No such file or directory: '../results/pmid-ena-results-2026-05-18T00-45-37-328Z.csv'

In [ ]:
# fetch abstracts for all papers that have accessions
has_accessions = mmc_df[mmc_df["accession_count"] > 0].copy()

abstracts = {}
for _, row in has_accessions.iterrows():
    pmid = str(row["pmid"])
    abstract = fetch_abstract_from_pmid(pmid)
    abstracts[pmid] = abstract
    print(f"PMID {pmid}: {'FOUND' if abstract else 'NOT FOUND'}")
    time.sleep(0.3)

found = sum(1 for a in abstracts.values() if a)
print(f"\nAbstracts found: {found}/{len(has_accessions)}")

PMID 42121260: FOUND
PMID 42120383: FOUND
PMID 42111294: FOUND
PMID 42106836: FOUND
PMID 42096904: FOUND
PMID 42068031: FOUND
PMID 42067917: FOUND
PMID 42063196: FOUND
PMID 42039826: FOUND
PMID 42019469: FOUND
PMID 41996550: FOUND
PMID 41994282: FOUND
PMID 41971320: FOUND
PMID 41969653: FOUND
PMID 41966829: FOUND
PMID 41928361: FOUND
PMID 41925227: FOUND
PMID 41910137: FOUND
PMID 41904571: FOUND
PMID 41897913: FOUND
PMID 41896653: FOUND
PMID 41892210: FOUND
PMID 41882801: FOUND
PMID 41877288: FOUND
PMID 41877267: FOUND
PMID 41872963: FOUND
PMID 41872905: FOUND
PMID 41862790: FOUND
PMID 41852415: FOUND
PMID 41841524: FOUND
PMID 41840712: FOUND
PMID 41829938: FOUND
PMID 41828538: FOUND
PMID 41827064: FOUND
PMID 41787302: FOUND
PMID 41784805: FOUND
PMID 41784373: FOUND
PMID 41778161: FOUND
PMID 41777544: FOUND
PMID 41776697: FOUND
PMID 41768933: FOUND
PMID 41759554: FOUND
PMID 41757098: FOUND
PMID 41751034: FOUND
PMID 41736978: FOUND
PMID 41736110: FOUND
PMID 41715166: FOUND
PMID 41714980

In [ ]:
# build accession -> abstract mapping
acc_to_abstract = {}
for _, row in mmc_df[mmc_df["accession_count"] > 0].iterrows():
    pmid = str(row["pmid"])
    abstract = abstracts.get(pmid)
    for acc in str(row["accessions"]).split(";"):
        acc = acc.strip()
        if re.match(r'PRJ[ENDB][A-Z]\d+', acc):
            acc_to_abstract[acc] = abstract

# add abstract column to combined catalog
combined_df["abstract"] = combined_df["study_accession"].map(acc_to_abstract)

# check coverage
print(f"Total studies: {len(combined_df)}")
print(f"Studies with abstract: {combined_df['abstract'].notna().sum()}")
print(f"Studies without abstract: {combined_df['abstract'].isna().sum()}")

Total studies: 657
Studies with abstract: 215
Studies without abstract: 442


In [ ]:
combined_df.to_csv("../results/combined_catalog.tsv", sep="\t", index=False)
print(f"Saved {len(combined_df)} studies to combined_catalog.tsv")

Saved 657 studies to combined_catalog.tsv


In [ ]:
import pandas as pd
import re

# accession -> pmid batch 2 results
mmc_batch2 = pd.read_csv("../results/pmid-ena-results-2026-05-19-batch2.csv")  

print(f"Batch 2 papers: {len(mmc_batch2)}")
print(f"Papers with accessions: {(mmc_batch2['accession_count'] > 0).sum()}")

# extract new accessions not already in combined_df
existing_accessions = set(combined_df["study_accession"].tolist())

new_accessions_batch2 = []
for _, row in mmc_batch2[mmc_batch2["accessions"].notna()].iterrows():
    for acc in str(row["accessions"]).split(";"):
        acc = acc.strip()
        if re.match(r'PRJ[ENDB][A-Z]\d+', acc) and acc not in existing_accessions:
            new_accessions_batch2.append(acc)

new_accessions_batch2 = list(set(new_accessions_batch2))
print(f"New unique accessions: {len(new_accessions_batch2)}")

Batch 2 papers: 500
Papers with accessions: 223
New unique accessions: 317


In [ ]:
new_results_batch2 = []

for i, study in enumerate(new_accessions_batch2):
    if i % 20 == 0:
        print(f"Progress: {i}/{len(new_accessions_batch2)}")
    
    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue
    
    new_results_batch2.append({
        "study_accession": study,
        "host_species": resolve_host_species(runs_df),
        "host_tax_id": runs_df["host_tax_id"].dropna().mode()[0] if len(runs_df["host_tax_id"].dropna()) > 0 else None,
        "body_site": runs_df["host_body_site"].dropna().mode()[0] if len(runs_df["host_body_site"].dropna()) > 0 else "",
        "country": runs_df["country"].dropna().mode()[0] if len(runs_df["country"].dropna()) > 0 else "",
        "n_samples": runs_df["sample_accession"].nunique(),
        "library_strategy": runs_df["library_strategy"].dropna().mode()[0] if len(runs_df["library_strategy"].dropna()) > 0 else None,
    })
    time.sleep(0.3)

print(f"\nFetched metadata for {len(new_results_batch2)} new studies")

Progress: 0/317
No runs found for PRJNA1023627
No runs found for PRJNA923293
No runs found for PRJNA115737
No runs found for PRJNA923758
No runs found for PRJNA910333
Progress: 20/317
Progress: 40/317
No runs found for PRJNA894235
No runs found for PRJNA843204
No runs found for PRJNA863131
No runs found for PRJDB13917
Progress: 60/317
No runs found for PRJNA290381
No runs found for PRJNA761907
No runs found for PRJNA725502
No runs found for PRJEB81441
Progress: 80/317
No runs found for PRJNA836733
No runs found for PRJNA1023577
Progress: 100/317
No runs found for PRJNA240272
No runs found for PRJEB35945
Progress: 120/317
No runs found for PRJNA923757
No runs found for PRJNA826263
Progress: 140/317
No runs found for PRJNA1065120
No runs found for PRJEB871798
No runs found for PRJEB35610
Progress: 160/317
No runs found for PRJEB1046
No runs found for PRJNA1077584
No runs found for PRJNA925792
Progress: 180/317
Error fetching PRJNA1145594: ('Connection aborted.', ConnectionResetError(54, 

In [ ]:
import pandas as pd
import re

# load batch 3+4
mmc_batch34 = pd.read_csv("../results/pmid-ena-results-2026-05-19-batch3.csv")  # adjust filename if different

print(f"Batch 3+4 papers: {len(mmc_batch34)}")
print(f"Papers with accessions: {(mmc_batch34['accession_count'] > 0).sum()}")

# extract new accessions not already in combined_df
existing_accessions = set(combined_df["study_accession"].tolist())

new_accessions_batch34 = []
for _, row in mmc_batch34[mmc_batch34["accessions"].notna()].iterrows():
    for acc in str(row["accessions"]).split(";"):
        acc = acc.strip()
        if re.match(r'PRJ[ENDB][A-Z]\d+', acc) and acc not in existing_accessions:
            new_accessions_batch34.append(acc)

new_accessions_batch34 = list(set(new_accessions_batch34))
print(f"New unique accessions not in catalog: {len(new_accessions_batch34)}")

Batch 3+4 papers: 846
Papers with accessions: 310


NameError: name 'combined_df' is not defined

In [11]:
# save current combined catalog
combined_df.to_csv("../results/combined_catalog.tsv", sep="\t", index=False)
print(f"Saved {len(combined_df)} studies")

# save abstracts dictionary
import json
with open("../results/abstracts.json", "w") as f:
    json.dump(abstracts, f)
print(f"Saved {len(abstracts)} abstracts")

NameError: name 'combined_df' is not defined